# ASR — 300h retrain

Extends the finished 100h LibriSpeech-only model (dev-clean greedy WER
10.1% / CER 2.8%; +KenLM 5.1% / 1.8%) with accent, spontaneous-speech and
channel/noise robustness, for a **live demo**: laptop microphone, in a room,
non-native English speaker. The failure mode being targeted is domain shift
(the current model has only ever heard clean read studio speech), not
insufficient data.

Architecture is UNCHANGED from `ablation_engine.py`: frozen `mHuBERT-147` +
LoRA (layers 1–12, r=16, alpha=32, `q_proj`/`v_proj`) + weighted-sum over
hidden-state layers + 2-layer MLP + CTC, character vocabulary
`A-Z, ', |, [UNK], [PAD]` (identical to `kenlm_grid.py`'s KenLM alphabet).

This notebook:
1. sets up a Python 3.11 `uv` environment (molab ships 3.13; KenLM does not
   build on 3.13, and 3.13 conflicts with the pinned stack here),
2. materialises the pipeline modules (`prepare_data.py`, `augment.py`,
   `build_cache.py`, `train_asr.py`, `eval_asr.py`) via `%%writefile`,
3. builds the 300h manifest (100 LibriSpeech + 106 Common Voice + 50 AMI +
   44 VCTK), with `assert_no_l2arctic` as a hard gate — L2-ARCTIC is the
   held-out OOD test set and is NEVER touched here,
4. runs the 50h high-LR probe (3 weighted-sum arms, augmentation ON),
5. runs the full 300h training with per-epoch checkpointing,
6. evaluates on the headline **two-column** table: dev-clean (comparability
   anchor) × L2-ARCTIC (OOD), greedy and +KenLM, against Whisper base/small/medium,
   with CPU/GPU RTF and peak RAM.


## 1 · Environment — Python 3.11 uv venv (pinned cu128 stack)

In [ ]:
import subprocess

# molab ships Python 3.13 + uv. We create a SEPARATE 3.11 venv because:
#   - 3.13 causes dependency conflicts with the pinned stack below
#   - KenLM does not build on 3.13 at all -- building here on 3.11 means KenLM
#     works on the SAME machine as training, removing the old dependency on
#     a separate Colab environment just for decoding.
subprocess.run(["uv", "venv", "--python", "3.11", "/marimo/asr311"], check=True)
subprocess.run(["/marimo/asr311/bin/python", "-m", "pip", "--version"], check=True)


In [ ]:
# torch / torchaudio / torchvision MUST be the SAME version from the SAME
# cu128 index. A mismatch previously caused a torchvision::nms crash that
# took down transformers entirely -- this is a known trap on this stack, not
# a hypothetical one, so the three are pinned together explicitly here.
TORCH_VER = "2.6.0"
CU_INDEX = "https://download.pytorch.org/whl/cu128"
PY311_BIN = "/marimo/asr311/bin/python"

subprocess.run(["uv", "pip", "install", "--python", PY311_BIN,
                 f"torch=={TORCH_VER}", f"torchaudio=={TORCH_VER}", f"torchvision=={TORCH_VER}",
                 "--index-url", CU_INDEX], check=True)

subprocess.run(["uv", "pip", "install", "--python", PY311_BIN,
                 "transformers>=4.44", "peft>=0.11", "jiwer", "soundfile==0.14.0",
                 "huggingface-hub>=0.24", "datasets==5.0.0", "numpy", "pandas", "tqdm",
                 "psutil", "pyctcdecode", "kenlm"], check=True)


In [ ]:
import subprocess, sys, os
from pathlib import Path

PY311 = "/marimo/asr311/bin/python"
ASR_DIR = Path("/marimo/asr")          # where the %%writefile cells below land
DATA_DIR = Path("/marimo/data")
CACHE_DIR = Path("/marimo/cache")
RUNS_DIR = Path("/marimo/runs")
NOISE_DIR = Path("/marimo/noise")      # MUSAN + DEMAND wavs
RIR_DIR = Path("/marimo/rir")          # OpenSLR-28 RIR wavs
for d in (ASR_DIR, DATA_DIR, CACHE_DIR, RUNS_DIR, NOISE_DIR, RIR_DIR):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(ASR_DIR)

def run(*args, cwd=ASR_DIR):
    """Run a step in the 3.11 venv as a subprocess, streaming output live.
    Training/eval run OUTSIDE the notebook kernel's own (3.13) interpreter."""
    print("+", " ".join(str(a) for a in args), flush=True)
    p = subprocess.run([PY311, *[str(a) for a in args]], cwd=str(cwd))
    if p.returncode != 0:
        raise RuntimeError(f"step failed: {args} (exit {p.returncode})")


## 2 · Pipeline modules (`%%writefile`)

These materialise the same files documented in `asr/README.md`. Each is also runnable standalone.

In [ ]:
%%writefile prepare_data.py
# /// script
# requires-python = ">=3.11"
# dependencies = [
#     "datasets==5.0.0",
#     "huggingface-hub>=0.24",
#     "soundfile==0.14.0",
#     "numpy",
#     "pandas",
#     "tqdm",
# ]
# ///
"""
ASR -- 300h retrain, data pipeline.

Builds the training manifest from four sources:
    LibriSpeech train-clean-100   100 h   comparability anchor (unchanged)
    Common Voice 22 EN            106 h   accent-stratified sample
    AMI (ihm+sdm, disjoint mtgs)   50 h   accent/spontaneous/far-field
    VCTK                           44 h   accent, studio-clean
                                  -----
                                  300 h

L2-ARCTIC IS NEVER TOUCHED HERE. It is the held-out OOD test set used only
in evaluation (see asr_300h.ipynb). assert_no_l2arctic() below is a
hard gate -- every manifest-writing path runs through it before the combined
manifest is written.

Text is normalised to the EXACT character vocabulary used by ablation_engine.py /
kenlm_grid.py:
    A-Z, ' (apostrophe), | (word separator), [UNK], [PAD] (=CTC blank)
No new characters are ever added to that vocabulary -- the existing KenLM
3-gram was built on LibriSpeech-normalised text and does not know any other
symbol. Rows that still contain out-of-vocabulary characters after
normalisation are dropped, and the drop rate is logged per corpus.

Usage:
    python prepare_data.py --corpus librispeech --out /marimo/data
    python prepare_data.py --corpus common_voice --out /marimo/data
    python prepare_data.py --corpus ami --out /marimo/data
    python prepare_data.py --corpus vctk --out /marimo/data
    python prepare_data.py --combine --out /marimo/data
"""

from __future__ import annotations

import argparse
import csv
import json
import os
import random
import re
import sys
import tarfile
import time
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np

# ============================================================================
# 0 . Vocabulary (must match ablation_engine.py / kenlm_grid.py exactly)
# ============================================================================

CHARS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ'")


def build_vocab() -> dict:
    v = {c: i for i, c in enumerate(CHARS)}
    v["|"], v["[UNK]"], v["[PAD]"] = len(v), len(v) + 1, len(v) + 2
    return v


VOCAB = build_vocab()
_ALLOWED = set(CHARS) | {" "}  # space becomes "|" downstream; both are legal here

# digits -> words, so "911" survives normalisation instead of being dropped as OOV
_DIGIT_WORDS = {
    "0": "ZERO", "1": "ONE", "2": "TWO", "3": "THREE", "4": "FOUR",
    "5": "FIVE", "6": "SIX", "7": "SEVEN", "8": "EIGHT", "9": "NINE",
}

_WS_RE = re.compile(r"\s+")


def normalize_text(raw: str) -> str | None:
    """Normalise to the CTC/KenLM character set. Returns None if the row is
    unrecoverable (still has OOV chars after normalisation, or is empty).

    Rules (per spec):
      - expand digits to words (911 -> NINE ONE ONE)
      - strip hyphens and periods (AMI spells letters as "S. S. H.")
      - uppercase
      - keep apostrophes
      - collapse whitespace
      - drop rows still containing OOV chars after all of the above

    IMPORTANT: only hyphens/periods are explicitly stripped. Any OTHER
    character outside A-Z/'/space causes the whole row to be DROPPED (return
    None), not silently deleted -- silently deleting stray punctuation would
    let rows with real OOV content (accented letters, stray symbols) sail
    through as if they were clean, which is exactly the failure mode the
    "drop rows still containing OOV chars" rule exists to prevent.
    """
    if not raw:
        return None
    s = raw.upper()
    # digit expansion BEFORE stripping punctuation, so "9-1-1" and "9.1.1."
    # both become "NINE ONE ONE" rather than "911" surviving as a bare token
    s = "".join(f" {_DIGIT_WORDS[ch]} " if ch in _DIGIT_WORDS else ch for ch in s)
    s = s.replace("-", " ").replace(".", " ")
    s = _WS_RE.sub(" ", s).strip()
    if not s:
        return None
    if any(c not in _ALLOWED for c in s):
        return None
    return s


# ============================================================================
# 1 . L2-ARCTIC leakage gate
# ============================================================================

_L2ARCTIC_MARKERS = ("l2-arctic", "l2_arctic", "l2arctic")


def assert_no_l2arctic(rows: list[dict], manifest_path) -> None:
    """Hard gate: L2-ARCTIC must NEVER appear in a training manifest.
    Checked on every row's corpus/source/audio_path field, and on the
    manifest path itself. Raises AssertionError (not a warning) on any hit."""
    mp = str(manifest_path).lower()
    assert not any(m in mp for m in _L2ARCTIC_MARKERS), (
        f"L2-ARCTIC marker found in manifest PATH itself: {manifest_path}. "
        "L2-ARCTIC is the held-out OOD test set and must never be written "
        "into a training manifest."
    )
    for r in rows:
        blob = " ".join(str(r.get(k, "")) for k in ("corpus", "source", "audio_path", "speaker"))
        blob = blob.lower()
        assert not any(m in blob for m in _L2ARCTIC_MARKERS), (
            f"L2-ARCTIC marker found in a training row: {r}. Aborting write "
            f"of {manifest_path} -- this would leak the OOD test set into training."
        )


# ============================================================================
# 2 . Manifest I/O helpers
# ============================================================================


def write_manifest(rows: list[dict], path: Path) -> None:
    assert_no_l2arctic(rows, path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")


def hours(rows: list[dict]) -> float:
    return sum(r["duration_s"] for r in rows) / 3600.0


def log(*a):
    print(*a, flush=True)


# ============================================================================
# 3 . LibriSpeech train-clean-100 (comparability anchor -- kept at 100h,
#     identical source/split to the 100h baseline model so the two runs are
#     comparable on dev-clean)
# ============================================================================


def build_librispeech(out_dir: Path, limit_hours: float = 100.0) -> Path:
    from datasets import load_dataset

    log("[librispeech] loading openslr/librispeech_asr (clean, train.100)...")
    ds = load_dataset("openslr/librispeech_asr", "clean", split="train.100")

    rows, oov_drop, kept_s = [], 0, 0.0
    for i in range(len(ds)):
        row = ds[i]
        audio = row["audio"]
        dur = len(audio["array"]) / audio["sampling_rate"]
        text = normalize_text(row["text"])
        if text is None:
            oov_drop += 1
            continue
        rows.append({
            "corpus": "librispeech",
            "source": "openslr/librispeech_asr:train.100",
            "audio_path": audio.get("path") or f"librispeech#{i}",
            "hf_index": i,
            "text": text,
            "duration_s": dur,
            "speaker": str(row.get("speaker_id", "")),
        })
        kept_s += dur
        if kept_s / 3600.0 >= limit_hours:
            break

    n_total = len(rows) + oov_drop
    log(f"[librispeech] kept {hours(rows):.2f}h / {len(rows)} rows | "
        f"OOV drop rate {oov_drop / max(1, n_total):.4f} ({oov_drop} rows)")

    out = out_dir / "manifest_librispeech.jsonl"
    write_manifest(rows, out)
    return out


# ============================================================================
# 4 . Common Voice 22 EN -- accent-stratified, snapshot_download only
#     (datasets v4.0 removed script-based loaders; fsicoli/common_voice_22_0
#     ships a loading script and load_dataset() will fail outright. Also the
#     full repo is 578 GB across 100+ languages -- never pull more than the
#     English shards.)
# ============================================================================

CV_REPO = "fsicoli/common_voice_22_0"
CV_TARGET_HOURS = 106.0
CV_PER_ACCENT_CAP_HOURS = 12.0  # no single accent bucket may dominate the 106h


def _cv_accent_list(row: dict) -> list[str]:
    """accent field may be called 'accent' or 'accents', singular or comma
    separated. Normalise to a list of lowercase, stripped labels."""
    raw = row.get("accents") if row.get("accents") else row.get("accent")
    if not raw:
        return ["unknown"]
    parts = [p.strip().lower() for p in str(raw).split(",") if p.strip()]
    return parts or ["unknown"]


def download_common_voice_en(cache_dir: Path) -> Path:
    """Fetch ONLY the English shards of Common Voice 22 via snapshot_download
    (allow_patterns), never load_dataset(). Returns the local directory that
    contains the English validated.tsv + clip archives.

    NOTE: the exact repo layout could not be verified from this environment
    (no network access here). allow_patterns is intentionally broad/redundant
    across a few plausible layouts (transcript/en/*, audio/en/**, en/**) so
    that whichever one the repo actually uses is matched; run with
    HF_HUB_VERBOSITY=info the first time and inspect `local` if it pulls
    unexpected extra files, then tighten the patterns.
    """
    from huggingface_hub import snapshot_download

    log("[common_voice] snapshot_download, English shards only "
        "(repo is 578 GB total across 100+ languages -- do NOT pull more)")
    local = snapshot_download(
        repo_id=CV_REPO,
        repo_type="dataset",
        allow_patterns=[
            "transcript/en/*",
            "transcript/en.tsv",
            "*/en/*",           # covers repo layouts that nest audio/ per split
            "audio/en/**",
            "en/**",
            "*en_validated*",
            "*en_clips*",
        ],
        local_dir=str(cache_dir / "common_voice_en_raw"),
    )
    log(f"[common_voice] snapshot at {local}")
    return Path(local)


def _find_cv_tsv(root: Path) -> Path:
    cands = list(root.rglob("validated.tsv")) or list(root.rglob("*validated*.tsv"))
    if not cands:
        raise FileNotFoundError(
            f"No validated.tsv found under {root} -- Common Voice repo layout "
            "may have changed; inspect the snapshot manually and adjust "
            "allow_patterns / this lookup."
        )
    return cands[0]


def _find_cv_audio(root: Path, clip_name: str):
    hits = list(root.rglob(clip_name))
    return hits[0] if hits else None


def build_common_voice(out_dir: Path, cache_dir: Path, target_hours: float = CV_TARGET_HOURS,
                        per_accent_cap_hours: float = CV_PER_ACCENT_CAP_HOURS,
                        seed: int = 1337) -> Path:
    import soundfile as sf

    root = download_common_voice_en(cache_dir)
    tsv_path = _find_cv_tsv(root)
    log(f"[common_voice] parsing {tsv_path}")

    by_accent: dict[str, list[dict]] = defaultdict(list)
    with tsv_path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            accents = _cv_accent_list(row)
            # a row with multiple accents is filed under each -- sampling
            # still respects the per-accent cap since we draw independently
            # below; duplicate audio across buckets is fine, deduped at end
            for acc in accents:
                by_accent[acc].append(row)

    hist_raw = {k: len(v) for k, v in sorted(by_accent.items(), key=lambda kv: -len(kv[1]))}
    log(f"[common_voice] raw accent histogram (row counts, top 15): "
        f"{dict(list(hist_raw.items())[:15])}")

    rng = random.Random(seed)
    for acc in by_accent:
        rng.shuffle(by_accent[acc])

    # Round-robin across accent buckets so the natural US/England mass doesn't
    # crowd out the long tail: pull one clip at a time per accent, respecting
    # both the per-accent hour cap and the global target.
    picked: dict[str, dict] = {}         # path -> row  (dedup)
    picked_secs: dict[str, float] = defaultdict(float)
    cap_s = per_accent_cap_hours * 3600.0
    target_s = target_hours * 3600.0
    cursors = {acc: 0 for acc in by_accent}
    accents_order = list(by_accent.keys())
    total_s = 0.0
    oov_drop, dur_drop, missing_audio = 0, 0, 0

    while total_s < target_s and accents_order:
        progressed = False
        for acc in list(accents_order):
            if picked_secs[acc] >= cap_s:
                if acc in accents_order:
                    accents_order.remove(acc)
                continue
            idx = cursors[acc]
            bucket = by_accent[acc]
            if idx >= len(bucket):
                if acc in accents_order:
                    accents_order.remove(acc)
                continue
            row = bucket[idx]
            cursors[acc] += 1
            path = row.get("path") or row.get("filename") or row.get("sentence_id")
            if not path or path in picked:
                continue
            audio_file = _find_cv_audio(root, path)
            if audio_file is None:
                missing_audio += 1
                continue
            try:
                info = sf.info(str(audio_file))
                dur = info.frames / info.samplerate
            except Exception:
                missing_audio += 1
                continue
            if dur <= 0 or dur > 30.0:
                dur_drop += 1
                continue
            text = normalize_text(row.get("sentence") or row.get("text") or "")
            if text is None:
                oov_drop += 1
                continue
            picked[path] = {
                "corpus": "common_voice",
                "source": CV_REPO,
                "audio_path": str(audio_file),
                "text": text,
                "duration_s": dur,
                "accents": _cv_accent_list(row),
                "speaker": row.get("client_id", ""),
            }
            picked_secs[acc] += dur
            total_s += dur
            progressed = True
            if total_s >= target_s:
                break
        if not progressed:
            break  # exhausted every bucket before hitting the target

    rows = list(picked.values())
    hist_kept = Counter()
    for r in rows:
        for acc in r["accents"]:
            hist_kept[acc] += 1
    log(f"[common_voice] KEPT accent histogram: {dict(hist_kept.most_common(20))}")
    log(f"[common_voice] kept {hours(rows):.2f}h / {len(rows)} rows "
        f"(target {target_hours}h, per-accent cap {per_accent_cap_hours}h) | "
        f"OOV drop {oov_drop} | duration-filter drop {dur_drop} | "
        f"missing audio {missing_audio}")

    out = out_dir / "manifest_common_voice.jsonl"
    write_manifest(rows, out)
    (out_dir / "common_voice_accent_histogram.json").write_text(
        json.dumps({"raw": hist_raw, "kept": dict(hist_kept.most_common())}, indent=2))
    return out


# ============================================================================
# 5 . AMI -- ihm (25h) + sdm (25h), DISJOINT meetings, four filters
# ============================================================================

AMI_REPO = "edinburghcstr/ami"
AMI_TARGET_HOURS_PER_MIC = 25.0

# (a) CTC feasibility: backbone runs at 50 frames/sec; CTC needs >=2 frames
# per target char (blank-separated repeats). duration_s*50 >= 2*len(text)
# i.e. roughly 40ms/char. This is the filter that PREVENTS crashes (inf/nan
# CTC loss from single-frame 0.02s clips) -- mandatory, not optional.
AMI_FRAMES_PER_SEC = 50.0
AMI_MIN_FRAMES_PER_CHAR = 2.0

# (b) pure filler stoplist -- backchannels with no lexical content.
# Entries are written with hyphens for readability, but normalize_text()
# turns "-" into a space (AMI also spells letters as "S. S. H." the same
# way), so the stoplist is matched against a hyphen-free/space-free form of
# both the stoplist and the candidate text -- otherwise "MM-HMM" in this set
# would never match the normalised "MM HMM" and the filter would silently
# no-op on exactly the entries that need the hyphen written out.
AMI_FILLER_STOPLIST = {
    "MM", "MMM", "HMM", "HM", "UH", "UM", "ERM", "AH", "OH", "EH",
    "MM-HMM", "UH-HUH",
}
_AMI_FILLER_COMPACT = {w.replace("-", "").replace(" ", "") for w in AMI_FILLER_STOPLIST}

# (c) short real words MUST survive (demo will contain them; they teach the
# model to emit little when little was said, i.e. anti-hallucination signal).
# Never zero them out -- only subsample to this keep-rate so they don't
# dominate the corpus. Named constant per spec.
AMI_SHORT_WORD_KEEP_RATE = 0.30
AMI_SHORT_WORD_MAX_CHARS = 12  # heuristic: "single short word" ballpark

# (d) character-rate sanity -- catches truncated automatic-alignment segments
AMI_MIN_CPS = 2.0
AMI_MAX_CPS = 25.0


def _ami_passes_filters(text: str, duration_s: float, rng: random.Random,
                         stats: Counter) -> bool:
    # normalise AMI's uppercase/apostrophe text through the shared normaliser
    # first so downstream stats operate on the same string CTC will train on
    n = normalize_text(text)
    if n is None:
        stats["oov"] += 1
        return False

    # (a) CTC feasibility -- mandatory
    if duration_s * AMI_FRAMES_PER_SEC < AMI_MIN_FRAMES_PER_CHAR * len(n.replace(" ", "")):
        stats["ctc_infeasible"] += 1
        return False

    # (b) pure filler stoplist (compared hyphen/space-insensitively -- see
    # _AMI_FILLER_COMPACT comment above)
    if n.replace(" ", "") in _AMI_FILLER_COMPACT:
        stats["filler"] += 1
        return False

    # (d) character-rate sanity (do this before the short-word keep so short
    # AND garbled segments are dropped for the right reason)
    cps = len(n) / max(duration_s, 1e-6)
    if not (AMI_MIN_CPS <= cps <= AMI_MAX_CPS):
        stats["bad_char_rate"] += 1
        return False

    # (c) keep short real words, but subsample so they don't dominate
    n_words = len(n.split())
    if n_words <= 2 and len(n) <= AMI_SHORT_WORD_MAX_CHARS:
        if rng.random() > AMI_SHORT_WORD_KEEP_RATE:
            stats["short_word_subsampled"] += 1
            return False
        stats["short_word_kept"] += 1
        return True

    stats["kept_normal"] += 1
    return True


def build_ami(out_dir: Path, cache_dir: Path,
              hours_per_mic: float = AMI_TARGET_HOURS_PER_MIC, seed: int = 1337) -> Path:
    from datasets import load_dataset

    rng = random.Random(seed)

    log("[ami] loading edinburghcstr/ami (parquet-native, both mic configs)...")
    ds_ihm = load_dataset(AMI_REPO, "ihm", split="train")
    ds_sdm = load_dataset(AMI_REPO, "sdm", split="train")

    # Partition meeting_ids so the SAME meeting never appears via both mics --
    # otherwise we get the same transcript twice with different audio, which
    # both inflates hours-counted and correlates train/dev if a meeting is
    # split across mic type by accident.
    meetings_ihm = sorted(set(ds_ihm["meeting_id"]))
    meetings_sdm = sorted(set(ds_sdm["meeting_id"]))
    all_meetings = sorted(set(meetings_ihm) | set(meetings_sdm))
    rng.shuffle(all_meetings)
    half = len(all_meetings) // 2
    ihm_meetings = set(all_meetings[:half])
    sdm_meetings = set(all_meetings[half:])
    assert ihm_meetings.isdisjoint(sdm_meetings), "AMI meeting partition is not disjoint"

    def _collect(ds, allowed_meetings, mic_tag, budget_hours):
        rows, stats, kept_s = [], Counter(), 0.0
        idxs = list(range(len(ds)))
        rng.shuffle(idxs)
        for i in idxs:
            if kept_s / 3600.0 >= budget_hours:
                break
            row = ds[i]
            if row["meeting_id"] not in allowed_meetings:
                continue
            audio = row["audio"]
            dur = row.get("end_time", 0.0) - row.get("begin_time", 0.0)
            if dur <= 0:
                dur = len(audio["array"]) / audio["sampling_rate"]
            text = row["text"]
            if not _ami_passes_filters(text, dur, rng, stats):
                continue
            norm = normalize_text(text)
            rows.append({
                "corpus": "ami",
                "source": f"{AMI_REPO}:{mic_tag}",
                "audio_path": audio.get("path") or f"ami_{mic_tag}#{i}",
                "hf_index": i,
                "mic": mic_tag,
                "meeting_id": row["meeting_id"],
                "speaker": row.get("speaker_id", ""),
                "text": norm,
                "duration_s": dur,
            })
            kept_s += dur
        return rows, stats

    rows_ihm, stats_ihm = _collect(ds_ihm, ihm_meetings, "ihm", hours_per_mic)
    rows_sdm, stats_sdm = _collect(ds_sdm, sdm_meetings, "sdm", hours_per_mic)

    used_meetings_ihm = {r["meeting_id"] for r in rows_ihm}
    used_meetings_sdm = {r["meeting_id"] for r in rows_sdm}
    assert used_meetings_ihm.isdisjoint(used_meetings_sdm), (
        "AMI disjointness violated after collection -- same meeting_id ended "
        "up on both ihm and sdm sides."
    )

    rows = rows_ihm + rows_sdm
    log(f"[ami] ihm: kept {hours(rows_ihm):.2f}h / {len(rows_ihm)} rows | filters={dict(stats_ihm)}")
    log(f"[ami] sdm: kept {hours(rows_sdm):.2f}h / {len(rows_sdm)} rows | filters={dict(stats_sdm)}")
    log(f"[ami] TOTAL kept {hours(rows):.2f}h / {len(rows)} rows | "
        f"disjoint meetings: ihm={len(used_meetings_ihm)} sdm={len(used_meetings_sdm)}")

    out = out_dir / "manifest_ami.jsonl"
    write_manifest(rows, out)
    return out


# ============================================================================
# 6 . VCTK -- accent, studio-clean, standard HF loader
# ============================================================================

VCTK_TARGET_HOURS = 44.0


def build_vctk(out_dir: Path, target_hours: float = VCTK_TARGET_HOURS) -> Path:
    from datasets import load_dataset

    log("[vctk] loading VCTK...")
    # HF mirrors vary in repo id; try the common ones in order. Verify the
    # correct one on huggingface.co before a real run (no network here).
    ds = None
    for repo in ("CSTR-Edinburgh/vctk", "vctk", "sanchit-gandhi/vctk"):
        try:
            ds = load_dataset(repo, split="train")
            log(f"[vctk] loaded from {repo}")
            break
        except Exception as e:
            log(f"[vctk] {repo} failed ({type(e).__name__}) -- trying next")
    if ds is None:
        raise RuntimeError(
            "Could not load VCTK from any known HF repo id. Verify the "
            "correct repo id on huggingface.co and edit the `repo` list above."
        )

    rows, oov_drop, kept_s = [], 0, 0.0
    idxs = list(range(len(ds)))
    random.Random(1337).shuffle(idxs)
    for i in idxs:
        if kept_s / 3600.0 >= target_hours:
            break
        row = ds[i]
        audio = row["audio"]
        dur = len(audio["array"]) / audio["sampling_rate"]
        text_field = row.get("text") or row.get("sentence") or ""
        text = normalize_text(text_field)
        if text is None:
            oov_drop += 1
            continue
        rows.append({
            "corpus": "vctk",
            "source": "vctk",
            "audio_path": audio.get("path") or f"vctk#{i}",
            "hf_index": i,
            "speaker": str(row.get("speaker_id", row.get("speaker", ""))),
            "text": text,
            "duration_s": dur,
        })
        kept_s += dur

    n_total = len(rows) + oov_drop
    log(f"[vctk] kept {hours(rows):.2f}h / {len(rows)} rows | "
        f"OOV drop rate {oov_drop / max(1, n_total):.4f}")

    out = out_dir / "manifest_vctk.jsonl"
    write_manifest(rows, out)
    return out


# ============================================================================
# 7 . Combine
# ============================================================================


def combine(out_dir: Path) -> Path:
    parts = ["manifest_librispeech.jsonl", "manifest_common_voice.jsonl",
             "manifest_ami.jsonl", "manifest_vctk.jsonl"]
    rows = []
    for p in parts:
        fp = out_dir / p
        if not fp.exists():
            log(f"[combine] WARNING: {fp} missing, skipping")
            continue
        with fp.open() as f:
            part_rows = [json.loads(l) for l in f if l.strip()]
        rows.extend(part_rows)
        log(f"[combine] {p}: {hours(part_rows):.2f}h / {len(part_rows)} rows")

    log(f"[combine] TOTAL {hours(rows):.2f}h / {len(rows)} rows "
        f"(target 300h: 100 librispeech + 106 common_voice + 50 ami + 44 vctk)")

    combined_path = out_dir / "manifest_combined.jsonl"
    write_manifest(rows, combined_path)

    by_corpus = Counter(r["corpus"] for r in rows)
    stats = {"total_hours": hours(rows), "total_rows": len(rows),
             "by_corpus_rows": dict(by_corpus),
             "by_corpus_hours": {c: hours([r for r in rows if r["corpus"] == c])
                                 for c in by_corpus}}
    (out_dir / "manifest_combined.stats.json").write_text(json.dumps(stats, indent=2))
    return combined_path


# ============================================================================
# 8 . CLI
# ============================================================================


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--corpus", choices=["librispeech", "common_voice", "ami", "vctk"])
    ap.add_argument("--combine", action="store_true")
    ap.add_argument("--out", default="/marimo/data")
    ap.add_argument("--cache", default="/marimo/cache")
    args = ap.parse_args()

    out_dir = Path(args.out)
    cache_dir = Path(args.cache)
    out_dir.mkdir(parents=True, exist_ok=True)
    cache_dir.mkdir(parents=True, exist_ok=True)

    t0 = time.time()
    if args.corpus == "librispeech":
        build_librispeech(out_dir)
    elif args.corpus == "common_voice":
        build_common_voice(out_dir, cache_dir)
    elif args.corpus == "ami":
        build_ami(out_dir, cache_dir)
    elif args.corpus == "vctk":
        build_vctk(out_dir)
    if args.combine:
        combine(out_dir)
    log(f"[done] {time.time() - t0:.0f}s")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile augment.py
# /// script
# requires-python = ">=3.11"
# dependencies = ["torch", "torchaudio"]
# [tool.uv.sources]
# torch = { index = "pytorch-cu128" }
# torchaudio = { index = "pytorch-cu128" }
# [[tool.uv.index]]
# name = "pytorch-cu128"
# url = "https://download.pytorch.org/whl/cu128"
# explicit = true
# ///
"""
ASR -- 300h retrain, GPU-side augmentation.

Runs on the BATCHED waveform tensor, on GPU, inside the training loop --
not per-sample on CPU in the DataLoader. The backbone is frozen (only LoRA +
weighted-sum + CTC head are trainable), so there is plenty of spare GPU
compute; doing this per-sample on CPU workers would starve the GPU instead.

Chain order matches the physical story used in aug_night_v2.py / ablation_engine.py
(source -> room -> noise -> channel -> spec masking): reversing it gives
wrong results (e.g. reverb AFTER the telephone round-trip is physically
backwards -- a room impulse response never happens inside a phone network).

What's reused from the existing CPU augmentation work (aug_night_v2.py,
aug_sweep_v1.py, ablation_engine.py):
  - the AugConfig-style dataclass of independent per-effect probabilities
    (p_clean escape hatch, p_speed, p_rir, p_noise, p_band/p_8k) and the
    SNR / T60 ranges those files already tuned (5-20 dB noise, 0.15-0.50 s
    T60 for reverb, 300-3400 Hz telephone band).
  - the physical ordering of the augmentation chain.
  - the "never always-on" philosophy (moderate probabilities, p_clean floor).
This module reimplements the *mechanics* with torch/torchaudio ops so the
whole chain runs batched on GPU; the numpy FFT versions in aug_night_v2.py /
ablation_engine.py were CPU, per-sample and are not reused verbatim for that reason.

Additions the SER work deliberately avoided but which apply here:
  - speed perturbation (0.9/1.0/1.1) -- SER avoided it because it corrupts
    emotion labels; that concern doesn't exist for ASR, where it's a
    standard, cheap accuracy win.
  - reverb is NOT down-weighted -- the live demo is a laptop mic in a room,
    not a phone line, so room acoustics matter more than channel/codec here.

Channel randomisation deliberately does NOT use torchaudio codec APIs
(io.AudioEffector / functional.apply_codec): torchaudio has been in
maintenance mode since 2.8 and encode/decode moved to TorchCodec, so those
APIs may simply be gone on molab's stack. A plain 16k->8k->16k
functional.resample round-trip gives the same "telephone bandwidth" effect
without any extra dependency.
"""

from __future__ import annotations

import glob
import math
import os
import random
from dataclasses import dataclass, field
from pathlib import Path

import torch
import torch.nn.functional as F
import torchaudio


# ============================================================================
# 1 . Noise / RIR bank -- loaded into RAM ONCE as fp32, shared across steps
# ============================================================================


class AudioBank:
    """Loads a folder of wav files into RAM once (~1.4 GB as fp32 for a few
    hundred MUSAN/DEMAND/OpenSLR-28 files at 16 kHz), and serves batched GPU
    tensors on demand. Kept as a plain python list of 1-D CPU tensors -- we
    only move the slice we need to GPU per batch, not the whole bank."""

    def __init__(self, root: str | None, sr: int = 16000, limit: int = 4000):
        self.sr = sr
        self.clips: list[torch.Tensor] = []
        if not root:
            return
        files = sorted(glob.glob(os.path.join(root, "**", "*.wav"), recursive=True))[:limit]
        total_s = 0.0
        for fp in files:
            try:
                w, s = torchaudio.load(fp)
            except Exception:
                continue
            w = w.mean(0) if w.dim() > 1 else w
            if s != sr:
                w = torchaudio.functional.resample(w, s, sr)
            self.clips.append(w.float())
            total_s += w.numel() / sr
        print(f"[BANK] {root}: {len(self.clips)} clips, {total_s / 3600:.2f} h loaded")

    def empty(self) -> bool:
        return len(self.clips) == 0

    def sample_batch(self, n: int, length: int, device, rng: random.Random) -> torch.Tensor:
        """Returns [n, length] float32 tensor on `device`, each row a random
        crop (looped if shorter than `length`) from a random bank clip."""
        out = torch.zeros(n, length, device=device)
        for i in range(n):
            c = self.clips[rng.randrange(len(self.clips))]
            if c.numel() < length:
                reps = math.ceil(length / max(1, c.numel()))
                c = c.repeat(reps)
            off = rng.randrange(0, max(1, c.numel() - length + 1))
            out[i] = c[off : off + length].to(device)
        return out


# ============================================================================
# 2 . Config -- mirrors the AugConfig shape from aug_night_v2.py / ablation_engine.py
# ============================================================================


@dataclass
class GPUAugConfig:
    p_clean: float = 0.35          # never-touch floor -- protects clean WER

    # speed perturbation (Ko et al. 2015) -- ENABLED here (unlike SER, which
    # avoided it to protect emotion labels; that reasoning is ASR-irrelevant)
    speed_rates: tuple = (0.9, 1.0, 1.1)
    p_speed: float = 0.6           # applied often; 1.0 is a no-op 1/3 of the time

    # reverb -- weighted UP, not down: the demo is a laptop mic in a ROOM
    p_rir: float = 0.35
    rir_dir: str | None = None     # OpenSLR-28 RIR wavs

    # additive noise -- MUSAN + DEMAND
    p_noise: float = 0.5
    snr_db: tuple = (5.0, 20.0)
    noise_dir: str | None = None

    # SpecAugment (Park et al. 2019) via torchaudio transforms, applied on a
    # complex STFT and inverted back to waveform (see apply_specaugment_gpu)
    p_specaug: float = 0.4
    freq_mask_param: int = 15
    time_mask_param: int = 35
    n_freq_masks: int = 2
    n_time_masks: int = 2

    # channel: 16k->8k->16k round trip via plain resample, NOT codec APIs
    # (io.AudioEffector / apply_codec may not exist on molab's torchaudio,
    # which has been in maintenance mode since 2.8 -- TorchCodec owns
    # encode/decode now). Also cheaper and dependency-free.
    p_channel_8k: float = 0.25

    _KEYS = ("p_speed", "p_rir", "p_noise", "p_specaug", "p_channel_8k")

    def any_on(self) -> bool:
        return any(getattr(self, k) > 0 for k in self._KEYS)


# ============================================================================
# 3 . Batched GPU ops
# ============================================================================


def _mix_at_snr(wave: torch.Tensor, noise: torch.Tensor, snr_db: torch.Tensor) -> torch.Tensor:
    """torchaudio.functional.add_noise wrapper -- both args [B, T], snr_db [B]."""
    return torchaudio.functional.add_noise(wave, noise, snr_db)


def apply_speed_gpu(wave: torch.Tensor, lengths: torch.Tensor, sr: int,
                     rates: tuple, rng: random.Random) -> tuple[torch.Tensor, torch.Tensor]:
    """One speed factor per UTTERANCE (not per batch): resample each row at
    its own factor, then re-pad the batch to the new max length. A no-op for
    rows that draw rate==1.0."""
    B, T = wave.shape
    out_rows, new_lens = [], []
    for i in range(B):
        rate = rng.choice(rates)
        w = wave[i, : lengths[i]]
        if rate != 1.0:
            # resample-based speed change: change the "declared" sample rate
            # by `rate`, then resample back to sr -> shortens/lengthens the
            # signal exactly like classic sox speed perturbation
            w = torchaudio.functional.resample(w.unsqueeze(0), int(sr * rate), sr).squeeze(0)
        out_rows.append(w)
        new_lens.append(w.numel())
    max_len = max(new_lens)
    out = torch.zeros(B, max_len, device=wave.device, dtype=wave.dtype)
    for i, w in enumerate(out_rows):
        out[i, : w.numel()] = w
    return out, torch.tensor(new_lens, device=wave.device, dtype=lengths.dtype)


def apply_rir_gpu(wave: torch.Tensor, bank: AudioBank, rng: random.Random) -> torch.Tensor:
    """Batched FFT convolution with a random RIR per row, using
    torchaudio.functional.fftconvolve (falls back to a manual torch.fft
    implementation if the installed torchaudio predates that function)."""
    B, T = wave.shape
    rirs = bank.sample_batch(B, min(T, 16000), wave.device, rng)  # cap RIR length ~1s
    # normalise each RIR to unit L1 energy so the reverberated signal doesn't
    # blow up in level (same convention as aug_rir in ablation_engine.py)
    rirs = rirs / (rirs.abs().sum(-1, keepdim=True) + 1e-9)
    try:
        wet = torchaudio.functional.fftconvolve(wave, rirs, mode="full")[:, :T]
    except AttributeError:
        n = T + rirs.shape[-1] - 1
        nfft = 1 << (n - 1).bit_length()
        W = torch.fft.rfft(wave, nfft)
        H = torch.fft.rfft(rirs, nfft)
        wet = torch.fft.irfft(W * H, nfft)[:, :T]
    return wet


def apply_noise_gpu(wave: torch.Tensor, lengths: torch.Tensor, bank: AudioBank,
                     snr_range: tuple, rng: random.Random) -> torch.Tensor:
    B, T = wave.shape
    noise = bank.sample_batch(B, T, wave.device, rng)
    snr = torch.empty(B, device=wave.device).uniform_(*snr_range)
    return _mix_at_snr(wave, noise, snr)


def apply_channel_8k_gpu(wave: torch.Tensor, sr: int = 16000) -> torch.Tensor:
    """16k -> 8k -> 16k round trip. Plain resample, no codec API (see module
    docstring for why codec APIs are avoided)."""
    down = torchaudio.functional.resample(wave, sr, sr // 2)
    back = torchaudio.functional.resample(down, sr // 2, sr)
    T = wave.shape[-1]
    if back.shape[-1] < T:
        back = F.pad(back, (0, T - back.shape[-1]))
    return back[..., :T]


def apply_specaugment_gpu(wave: torch.Tensor, cfg: GPUAugConfig, sr: int = 16000) -> torch.Tensor:
    """SpecAugment (Park et al. 2019) via torchaudio.transforms.FrequencyMasking
    / TimeMasking, applied on a complex STFT (torchaudio.transforms.Spectrogram
    with power=None) and inverted back to waveform with InverseSpectrogram.
    Because encode/decode is a matched STFT/ISTFT pair, this round-trip is
    lossless except in the masked bins/frames -- so the backbone still sees a
    raw waveform (as ablation_engine.py's CTC pipeline expects), not a spectrogram."""
    n_fft, hop = 400, 160  # 25ms / 10ms @ 16kHz, standard ASR STFT config
    spec_fn = torchaudio.transforms.Spectrogram(n_fft=n_fft, hop_length=hop,
                                                 power=None).to(wave.device)
    ispec_fn = torchaudio.transforms.InverseSpectrogram(n_fft=n_fft, hop_length=hop
                                                        ).to(wave.device)
    freq_mask = torchaudio.transforms.FrequencyMasking(cfg.freq_mask_param).to(wave.device)
    time_mask = torchaudio.transforms.TimeMasking(cfg.time_mask_param).to(wave.device)

    spec = spec_fn(wave)  # [B, F, T] complex
    for _ in range(cfg.n_freq_masks):
        spec = freq_mask(spec)
    for _ in range(cfg.n_time_masks):
        spec = time_mask(spec)
    out = ispec_fn(spec, length=wave.shape[-1])
    return out.real if out.is_complex() else out


# ============================================================================
# 4 . Top-level pipeline
# ============================================================================


class GPUAugmentPipeline:
    """Owns the noise/RIR banks and applies the full chain to a batch.

    Usage inside the training loop (batch already on GPU):
        aug = GPUAugmentPipeline(cfg, noise_dir=..., rir_dir=..., device=dev)
        X, wl = aug(X, wl)   # X: [B, T] float32 waveform, wl: [B] lengths
    """

    def __init__(self, cfg: GPUAugConfig, device: str, seed: int = 1337):
        self.cfg = cfg
        self.device = device
        self.rng = random.Random(seed)
        self.noise_bank = AudioBank(cfg.noise_dir)
        self.rir_bank = AudioBank(cfg.rir_dir)

    def __call__(self, wave: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        cfg = self.cfg
        if not cfg.any_on() or self.rng.random() < cfg.p_clean:
            return wave, lengths

        # 1. source: speed perturbation (changes length -> do first, before
        #    any effect that assumes a fixed T)
        if self.rng.random() < cfg.p_speed:
            wave, lengths = apply_speed_gpu(wave, lengths, 16000, cfg.speed_rates, self.rng)

        # 2. room: reverb
        if not self.rir_bank.empty() and self.rng.random() < cfg.p_rir:
            wave = apply_rir_gpu(wave, self.rir_bank, self.rng)

        # 3. noise: MUSAN/DEMAND additive noise at random SNR
        if not self.noise_bank.empty() and self.rng.random() < cfg.p_noise:
            wave = apply_noise_gpu(wave, lengths, self.noise_bank, cfg.snr_db, self.rng)

        # 4. channel: telephone-band round trip (plain resample, no codec API)
        if self.rng.random() < cfg.p_channel_8k:
            wave = apply_channel_8k_gpu(wave)

        # 5. SpecAugment -- time/frequency masking, applied last (regularises
        #    the representation the backbone actually consumes)
        if self.rng.random() < cfg.p_specaug:
            wave = apply_specaugment_gpu(wave, cfg)

        peak = wave.abs().amax(dim=-1, keepdim=True).clamp_min(1e-9)
        over = peak > 1.0
        wave = torch.where(over, wave / peak, wave)
        return wave, lengths


# ============================================================================
# 5 . Deterministic eval-time degradations (mirrors ablation_engine.py degrade_eval,
#     kept torch-native so the same eval harness can run on GPU batches)
# ============================================================================


def degrade_eval_gpu(wave: torch.Tensor, mode: str | None, sr: int = 16000) -> torch.Tensor:
    if mode in (None, "clean"):
        return wave
    if mode == "tel8k":
        return apply_channel_8k_gpu(wave, sr)
    if mode == "noisy":
        g = torch.Generator(device="cpu").manual_seed(12345)
        noise = torch.randn(wave.shape, generator=g).to(wave.device)
        snr = torch.full((wave.shape[0],), 10.0, device=wave.device)
        return _mix_at_snr(wave, noise, snr)
    raise ValueError(mode)


if __name__ == "__main__":
    # Minimal self-test -- runs on CPU, no bank files needed. Confirms the
    # pipeline is at least shape-correct end to end.
    torch.manual_seed(0)
    cfg = GPUAugConfig(p_clean=0.0, p_speed=1.0, p_rir=0.0, p_noise=0.0,
                       p_specaug=1.0, p_channel_8k=1.0)
    pipe = GPUAugmentPipeline(cfg, device="cpu")
    X = torch.randn(4, 16000)
    L = torch.tensor([16000, 12000, 8000, 4000])
    Y, YL = pipe(X, L)
    assert Y.shape[0] == X.shape[0]
    assert YL.max().item() <= Y.shape[1]
    print("[selftest] OK", Y.shape, YL.tolist())


In [ ]:
%%writefile build_cache.py
# /// script
# requires-python = ">=3.11"
# dependencies = ["numpy", "soundfile==0.14.0", "datasets==5.0.0"]
# ///
"""
ASR -- 300h retrain, packed-cache builder.

Converts manifest_combined.jsonl (produced by prepare_data.py) into the SAME
packed int16 binary + meta.json cache format ablation_engine.py's `prepare()`
uses (audio.i16 + offsets + texts), so the training script's Dataset class
can stay close to SpeechDS in ablation_engine.py.

Design decision (not specified verbatim in the task, made here for
consistency with the existing codebase): LibriSpeech and AMI rows carry an
`hf_index` into their HF dataset split rather than a standalone audio file
(that's how ablation_engine.py and the AMI loader consume them -- parquet-backed,
audio decoded from in-memory arrow arrays, no extracted wav files on disk).
Common Voice and VCTK rows carry a real `audio_path` on disk. This script
handles both: it re-opens the relevant HF dataset split once per corpus and
indexes into it for hf_index rows, and reads directly from disk for
audio_path rows.
"""

from __future__ import annotations

import hashlib
import io
import json
import time
from pathlib import Path

import numpy as np


def log(*a):
    print(*a, flush=True)


def _decode_soundfile(path: str, sr_target: int) -> np.ndarray:
    import soundfile as sf

    w, sr = sf.read(path, dtype="float32")
    w = np.asarray(w, np.float32)
    if w.ndim > 1:
        w = w.mean(1)
    if int(sr) != sr_target:
        w = np.interp(np.linspace(0, len(w) - 1, int(len(w) * sr_target / sr)),
                      np.arange(len(w)), w).astype(np.float32)
    return w


def _decode_hf_cell(cell, sr_target: int) -> np.ndarray:
    import soundfile as sf

    if isinstance(cell, dict) and cell.get("array") is not None:
        w, sr = np.asarray(cell["array"], np.float32), cell.get("sampling_rate", sr_target)
    elif isinstance(cell, dict) and cell.get("bytes"):
        w, sr = sf.read(io.BytesIO(cell["bytes"]), dtype="float32")
    elif isinstance(cell, dict) and cell.get("path"):
        w, sr = sf.read(cell["path"], dtype="float32")
    else:
        raise ValueError(f"unrecognised audio cell: {type(cell)}")
    w = np.asarray(w, np.float32)
    if w.ndim > 1:
        w = w.mean(1)
    if int(sr) != sr_target:
        w = np.interp(np.linspace(0, len(w) - 1, int(len(w) * sr_target / sr)),
                      np.arange(len(w)), w).astype(np.float32)
    return w


_HF_SPLIT_CACHE = {}


def _hf_split(corpus: str, source: str):
    key = (corpus, source)
    if key in _HF_SPLIT_CACHE:
        return _HF_SPLIT_CACHE[key]
    from datasets import load_dataset

    if corpus == "librispeech":
        ds = load_dataset("openslr/librispeech_asr", "clean", split="train.100")
    elif corpus == "ami":
        mic = source.split(":")[-1]
        ds = load_dataset("edinburghcstr/ami", mic, split="train")
    else:
        raise ValueError(f"no hf_index loader for corpus={corpus}")
    _HF_SPLIT_CACHE[key] = ds
    return ds


def build_cache(manifest_path: Path, cache_dir: Path, sr: int = 16000,
                 max_rows: int | None = None) -> Path:
    rows = [json.loads(l) for l in manifest_path.read_text().splitlines() if l.strip()]
    if max_rows:
        rows = rows[:max_rows]

    key = hashlib.md5(f"{manifest_path}|{len(rows)}|{sr}".encode()).hexdigest()[:12]
    d = cache_dir / f"combined_{key}"
    bin_p, meta_p = d / "audio.i16", d / "meta.json"
    if bin_p.exists() and meta_p.exists():
        log(f"[cache] found existing cache at {d}")
        return d

    d.mkdir(parents=True, exist_ok=True)
    t0, offs, texts, corpora, pos = time.perf_counter(), [0], [], [], 0
    with open(bin_p, "wb") as f:
        for i, row in enumerate(rows):
            try:
                if row.get("audio_path") and Path(row["audio_path"]).exists():
                    w = _decode_soundfile(row["audio_path"], sr)
                else:
                    ds = _hf_split(row["corpus"], row["source"])
                    cell = ds[row["hf_index"]]["audio"]
                    w = _decode_hf_cell(cell, sr)
            except Exception as e:
                log(f"[cache] WARN skip row {i} ({row.get('corpus')}): {type(e).__name__}: {e}")
                continue
            q = np.clip(np.rint(w * 32768.0), -32768, 32767).astype(np.int16)
            f.write(q.tobytes())
            pos += q.size
            offs.append(pos)
            texts.append(row["text"])
            corpora.append(row["corpus"])
            if (i + 1) % 2000 == 0:
                log(f"  [cache] {i + 1}/{len(rows)} ({pos * 2 / 1e9:.1f} GB, "
                    f"{time.perf_counter() - t0:.0f}s)")

    meta_p.write_text(json.dumps({"offsets": offs, "texts": texts, "corpora": corpora}))
    total_h = (offs[-1]) / sr / 3600.0
    log(f"[cache] {len(texts)} utterances, {total_h:.2f}h -> {d} "
        f"({(time.perf_counter() - t0) / 60:.1f} min)")
    return d


if __name__ == "__main__":
    import argparse

    ap = argparse.ArgumentParser()
    ap.add_argument("--manifest", default="/marimo/data/manifest_combined.jsonl")
    ap.add_argument("--cache", default="/marimo/cache")
    ap.add_argument("--max-rows", type=int, default=None)
    args = ap.parse_args()
    build_cache(Path(args.manifest), Path(args.cache), max_rows=args.max_rows)


In [ ]:
%%writefile train_asr.py
# /// script
# requires-python = ">=3.11"
# dependencies = [
#     "torch", "torchaudio", "transformers>=4.44", "peft>=0.11", "jiwer", "numpy",
# ]
# [tool.uv.sources]
# torch = { index = "pytorch-cu128" }
# torchaudio = { index = "pytorch-cu128" }
# [[tool.uv.index]]
# name = "pytorch-cu128"
# url = "https://download.pytorch.org/whl/cu128"
# explicit = true
# ///
"""
ASR -- 300h retrain, training script.

Reuses the architecture, LoRA config, weighted-sum head and CTC training
loop from ablation_engine.py VERBATIM (frozen mHuBERT-147 + LoRA on q_proj/v_proj
layers 1-12 + weighted-sum over configurable `ws` layers + 2-layer MLP CTC
head, AdamW with three param groups at different LRs, ReduceLROnPlateau on
CER, gradient accumulation, length-bucketed batching). What's NEW here:

  - data comes from the packed cache built by build_cache.py out of the
    300h combined manifest (prepare_data.py), instead of ablation_engine.py's
    LibriSpeech-only parquet cache.
  - augmentation is GPUAugmentPipeline (augment.py) applied to the batched
    GPU waveform tensor, instead of ablation_engine.py's per-sample CPU numpy aug.
  - the model is fully parameterised (--ws, --lora-layers, --lr-scale,
    --hours-subset) so this ONE script serves both the 50h probe (three WS
    arms, high LR) and the full 300h run -- avoiding a second, drifting copy
    of the training loop.
  - per-epoch checkpoints are kept as immutable ep{N:03d}.pt snapshots IN
    ADDITION to the resumable last.pt ablation_engine.py already writes: an ~8h
    unattended cloud run must survive a disconnect, and a single overwritten
    last.pt is one bad write away from losing everything.

Usage:
    python train_asr.py --run FINAL_300h --cache-dir /marimo/cache/combined_XXXX \
        --ws 9,10,11,12 --lora-layers 1-12 --epochs 30 --batch 64 --accum 4 \
        --noise-dir /marimo/noise --rir-dir /marimo/rir --out /marimo/runs

    # 50h probe, control arm:
    python train_asr.py --run probe_control --cache-dir ... --hours-subset 50 \
        --ws 9,10,11,12 --lora-layers 1-12 --epochs 6 --lr-scale 3.0

    # 50h probe, lower-layer arm:
    python train_asr.py --run probe_lowerA --cache-dir ... --hours-subset 50 \
        --ws 5,6,7,8 --lora-layers 1-12 --epochs 6 --lr-scale 3.0
"""

from __future__ import annotations

import argparse
import contextlib
import gc
import json
import sys
import time
from dataclasses import dataclass, field, asdict
from itertools import groupby
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
from prepare_data import build_vocab  # same vocab as ablation_engine.py / kenlm_grid.py
from augment import GPUAugConfig, GPUAugmentPipeline


def log(*a):
    print(*a, flush=True)


# ============================================================================
# Config
# ============================================================================


@dataclass
class Cfg:
    run: str = "run"
    ws: tuple = (9, 10, 11, 12)
    lora_layers: tuple = tuple(range(1, 13))
    lora_r: int = 16
    lora_alpha: int = 32
    hid: int = 768
    sr: int = 16000
    batch: int = 64
    batch_secs: float = 20.0
    accum: int = 4
    epochs: int = 30
    head_lr: float = 1e-3
    lora_lr: float = 2e-4
    w_lr: float = 1e-3
    lr_scale: float = 1.0          # multiplies all three LRs -- probe uses >1
    weight_decay: float = 0.0
    clip: float = 5.0
    patience: int = 4
    stop_patience: int = 12
    workers: int = 8
    seed: int = 1337
    hours_subset: float | None = None  # None = full cache; 50.0 for the probe
    aug_on: bool = True
    noise_dir: str | None = None
    rir_dir: str | None = None

    def __post_init__(self):
        self.ws = tuple(sorted(int(x) for x in self.ws))
        self.lora_layers = tuple(sorted(int(x) for x in self.lora_layers))


# ============================================================================
# Data: reads the packed int16 cache built by build_cache.py -- SAME format
# ablation_engine.py's prepare() uses (audio.i16 memmap + offsets + texts).
# ============================================================================


class SpeechDS:
    def __init__(self, cache_dir: Path, vocab: dict, sr: int, subset_hours: float | None = None,
                 seed: int = 1337):
        meta = json.loads((cache_dir / "meta.json").read_text())
        offs = np.asarray(meta["offsets"], dtype=np.int64)
        texts = meta["texts"]
        n = int(offs[-1])
        buf = np.memmap(cache_dir / "audio.i16", dtype=np.int16, mode="r", shape=(n,))

        if subset_hours is not None:
            lens = np.diff(offs)
            rng = np.random.default_rng(seed)
            order = rng.permutation(len(texts))
            budget = subset_hours * 3600.0 * sr
            keep, acc = [], 0.0
            for i in order:
                if acc >= budget:
                    break
                keep.append(int(i))
                acc += lens[i]
            keep = sorted(keep)
            self._idx = keep
        else:
            self._idx = list(range(len(texts)))

        self.buf, self.offs, self.texts, self.vocab, self.sr = buf, offs, texts, vocab, sr

    def __len__(self):
        return len(self._idx)

    def _raw(self, j):
        i = self._idx[j]
        a, b = int(self.offs[i]), int(self.offs[i + 1])
        return np.asarray(self.buf[a:b], np.float32) / 32768.0

    def text(self, j):
        return self.texts[self._idx[j]]

    def __getitem__(self, j):
        import torch

        w = self._raw(j)
        ids = [self.vocab.get(c, self.vocab["[UNK]"]) for c in self.text(j).replace(" ", "|")]
        return torch.from_numpy(np.ascontiguousarray(w)), torch.tensor(ids, dtype=torch.long), j


class LengthBucket:
    """Same frame-budget bucketing as ablation_engine.py's LengthBucket."""

    def __init__(self, lengths, batch, budget, shuffle=True, seed=0, pool_mult=50):
        self.L = np.asarray(lengths, np.int64)
        self.b, self.budget = batch, int(budget)
        self.shuffle, self.seed = shuffle, seed
        self.pool, self.epoch = batch * pool_mult, 0
        self._cache = self._build(0)

    def _build(self, epoch):
        g = np.random.default_rng(self.seed + epoch)
        idx = g.permutation(len(self.L)) if self.shuffle else np.arange(len(self.L))
        out, cur = [], []
        for i in range(0, len(idx), self.pool):
            ch = idx[i:i + self.pool]
            ch = ch[np.argsort(self.L[ch], kind="stable")]
            for j in ch:
                Lj = int(self.L[j])
                if cur and (len(cur) + 1 > self.b or Lj * (len(cur) + 1) > self.budget):
                    out.append(cur)
                    cur = [int(j)]
                else:
                    cur.append(int(j))
            if cur:
                out.append(cur)
                cur = []
        if cur:
            out.append(cur)
        if self.shuffle:
            g.shuffle(out)
        return out

    def __iter__(self):
        out = self._cache if self._cache is not None else self._build(self.epoch)
        self._cache = None
        self.epoch += 1
        self._n = len(out)
        return iter(out)

    def __len__(self):
        return len(self._cache) if self._cache is not None else getattr(self, "_n", 1)


def collate(batch, pad):
    import torch

    ws, ls, ix = zip(*batch)
    wl = torch.tensor([len(w) for w in ws])
    ll = torch.tensor([len(l) for l in ls])
    X = torch.zeros(len(ws), int(wl.max()))
    Y = torch.zeros(len(ls), int(ll.max()), dtype=torch.long)
    for i, (w, l) in enumerate(zip(ws, ls)):
        X[i, : len(w)] = w
        Y[i, : len(l)] = l
    return X, Y, wl, ll, torch.tensor(ix)


def make_loader(ds, cfg, shuffle):
    import torch
    from torch.utils.data import DataLoader

    lens = np.diff(ds.offs)[ds._idx]
    sampler = LengthBucket(lens, cfg.batch, cfg.batch * cfg.batch_secs * cfg.sr,
                            shuffle=shuffle, seed=cfg.seed)
    return DataLoader(ds, batch_sampler=sampler,
                       collate_fn=lambda b: collate(b, None),
                       num_workers=cfg.workers, pin_memory=True,
                       persistent_workers=cfg.workers > 0)


# ============================================================================
# Model -- verbatim from ablation_engine.py (backbone + LoRA + weighted-sum head)
# ============================================================================


def build_backbone(cfg: Cfg, device: str):
    import torch
    import torch.nn as nn
    from transformers import HubertModel
    from peft import LoraConfig, inject_adapter_in_model

    BACKBONE = "utter-project/mHuBERT-147"
    kw = dict(mask_time_prob=0.0, mask_feature_prob=0.0, apply_spec_augment=False,
              hidden_dropout=0.0, attention_dropout=0.0, activation_dropout=0.0,
              feat_proj_dropout=0.0, final_dropout=0.0, layerdrop=0.0)
    try:
        bb = HubertModel.from_pretrained(BACKBONE, attn_implementation="sdpa", **kw)
        log("[BB] attention: sdpa")
    except Exception as e:
        bb = HubertModel.from_pretrained(BACKBONE, **kw)
        log(f"[BB] attention: eager (sdpa unavailable: {type(e).__name__})")
    bb = bb.to(device)

    lora_cfg = LoraConfig(r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=0.0,
                          target_modules=["q_proj", "v_proj"], bias="none",
                          layers_to_transform=[i - 1 for i in cfg.lora_layers])
    bb = inject_adapter_in_model(lora_cfg, bb)
    for n, p in bb.named_parameters():
        p.requires_grad = "lora_" in n
    got = sum(p.numel() for p in bb.parameters() if p.requires_grad)
    exp = 2 * cfg.hid * cfg.lora_r * 2 * len(cfg.lora_layers)
    log(f"[LORA] {got:,} trainable params (expected {exp:,})")
    assert got == exp, f"LoRA out of scope: {got:,} != {exp:,}"
    bb.eval()  # backbone always in eval() -- SpecAugment/dropout are handled
               # by augment.py's GPUAugmentPipeline on the waveform instead
    return bb, bb._get_feat_extract_output_lengths


def make_head(cfg: Cfg, vocab_size: int, device: str):
    import torch
    import torch.nn as nn

    class Head(nn.Module):
        def __init__(self, n, dim, V):
            super().__init__()
            self.n = n
            self.layer_w = nn.Parameter(torch.zeros(n))
            self.net = nn.Sequential(nn.Linear(dim, dim), nn.ELU(), nn.Dropout(0.0),
                                     nn.Linear(dim, V))

        def weights(self):
            return self.layer_w.softmax(0)

        def forward(self, x):
            w = self.layer_w.softmax(0)
            f = (x * w[None, None, :, None]).sum(2)
            return self.net(f)

    return Head(len(cfg.ws), cfg.hid, vocab_size).to(device)


def decode_greedy(ids, i2c, blank, unk):
    return "".join(i2c.get(k, "") for k, _ in groupby(ids) if k not in (blank, unk)
                   ).replace("|", " ").strip()


# ============================================================================
# Train / eval loop
# ============================================================================


def evaluate(head, bb, dl, ds, flen, dev, i2c, blank, unk):
    import torch
    import jiwer

    head.eval()
    bb.eval()
    H, R = [], []
    with torch.no_grad():
        for X, _, wl, _, ix in dl:
            X = X.to(dev, non_blocking=True)
            am = (torch.arange(X.shape[1], device=dev)[None, :] < wl.to(dev)[:, None]).long()
            with torch.autocast(device_type=dev, dtype=torch.bfloat16):
                o = bb(X, attention_mask=am, output_hidden_states=True)
                cfg_ws = getattr(evaluate, "_ws", None)
            xl = flen(wl.to(dev))
            pr = head(torch.stack([o.hidden_states[L] for L in evaluate._ws], 2).float())
            pr = pr.argmax(-1).cpu().numpy()
            for b, j in enumerate(ix.tolist()):
                H.append(decode_greedy(pr[b, : int(xl[b])].tolist(), i2c, blank, unk))
                R.append(ds.text(j))
    return jiwer.wer(R, H), jiwer.cer(R, H)


def train_one(cfg: Cfg, out_root: Path, cache_dir: Path):
    import torch
    import torch.nn as nn

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(cfg.seed)
    if dev == "cuda":
        torch.cuda.reset_peak_memory_stats()
    torch.backends.cuda.matmul.allow_tf32 = True

    run_dir = out_root / cfg.run
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))
    log(f"[CFG] {cfg.run} | ws={list(cfg.ws)} | lora={list(cfg.lora_layers)} | "
        f"epochs={cfg.epochs} | lr_scale={cfg.lr_scale} | subset={cfg.hours_subset}h | "
        f"aug={'on' if cfg.aug_on else 'off'}")

    vocab = build_vocab()
    blank, unk = vocab["[PAD]"], vocab["[UNK]"]
    i2c = {v: k for k, v in vocab.items()}

    tr = SpeechDS(cache_dir, vocab, cfg.sr, subset_hours=cfg.hours_subset, seed=cfg.seed)
    # dev split: last 5% of the cache, held out from the subset draw above --
    # simple and reproducible given a fixed seed and a pre-shuffled manifest
    n_dev = max(1, int(0.05 * len(tr)))
    dv_idx = tr._idx[-n_dev:]
    tr._idx = tr._idx[:-n_dev]
    dv = SpeechDS.__new__(SpeechDS)
    dv.buf, dv.offs, dv.texts, dv.vocab, dv.sr = tr.buf, tr.offs, tr.texts, tr.vocab, tr.sr
    dv._idx = dv_idx

    tdl = make_loader(tr, cfg, True)
    ddl = make_loader(dv, cfg, False)
    log(f"[DATA] train={len(tr)} dev={len(dv)} utterances")

    bb, flen = build_backbone(cfg, dev)
    head = make_head(cfg, len(vocab), dev)
    evaluate._ws = cfg.ws

    aug_cfg = GPUAugConfig(noise_dir=cfg.noise_dir, rir_dir=cfg.rir_dir)
    if not cfg.aug_on:
        aug_cfg.p_clean = 1.0  # forces every batch through unmodified
    augmenter = GPUAugmentPipeline(aug_cfg, device=dev, seed=cfg.seed)

    groups = [
        {"params": [p for n, p in head.named_parameters() if n != "layer_w"],
         "lr": cfg.head_lr * cfg.lr_scale, "weight_decay": cfg.weight_decay},
        {"params": [head.layer_w], "lr": cfg.w_lr * cfg.lr_scale, "weight_decay": 0.0},
        {"params": [p for p in bb.parameters() if p.requires_grad],
         "lr": cfg.lora_lr * cfg.lr_scale, "weight_decay": cfg.weight_decay},
    ]
    opt = torch.optim.AdamW(groups, fused=(dev == "cuda"))
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, "min", factor=0.5,
                                                      patience=cfg.patience, threshold=0.005)
    ctc = nn.CTCLoss(blank=blank, reduction="mean", zero_infinity=True)
    trainable = [p for g in opt.param_groups for p in g["params"]]

    hist_path, last_path = run_dir / "history.jsonl", run_dir / "last.pt"
    ep0, best, best_ep, hist = 1, float("inf"), 0, []
    if last_path.exists():
        ck = torch.load(last_path, map_location=dev, weights_only=False)
        head.load_state_dict(ck["head"])
        bb.load_state_dict(ck["adapter"], strict=False)
        opt.load_state_dict(ck["opt"])
        with contextlib.suppress(Exception):
            sch.load_state_dict(ck["sch"])
        ep0, best, best_ep = ck["epoch"] + 1, ck["best"], ck["best_ep"]
        hist = ([json.loads(l) for l in hist_path.read_text().splitlines() if l.strip()]
                if hist_path.exists() else [])
        log(f"[RESUME] from epoch {ck['epoch']}, best CER {best * 100:.2f}%")
    if not hist_path.exists():
        hist_path.write_text("")

    for ep in range(ep0, cfg.epochs + 1):
        head.train()
        bb.eval()  # backbone stays frozen/eval; augmentation happens on the waveform
        t0, tot, nb = time.perf_counter(), 0.0, 0
        opt.zero_grad(set_to_none=True)
        for X, Y, wl, ll, _ in tdl:
            X, Y = X.to(dev, non_blocking=True), Y.to(dev, non_blocking=True)
            wl = wl.to(dev)
            X, wl = augmenter(X, wl)
            am = (torch.arange(X.shape[1], device=dev)[None, :] < wl[:, None]).long()
            with torch.autocast(device_type=dev, dtype=torch.bfloat16):
                o = bb(X, attention_mask=am, output_hidden_states=True)
                hs = torch.stack([o.hidden_states[L] for L in cfg.ws], 2)
            xl = flen(wl)
            lg = head(hs.float())
            loss = ctc(lg.log_softmax(-1).transpose(0, 1), Y, xl, ll.to(dev))
            (loss / cfg.accum).backward()
            nb += 1
            if nb % cfg.accum == 0:
                torch.nn.utils.clip_grad_norm_(trainable, cfg.clip)
                opt.step()
                opt.zero_grad(set_to_none=True)
            tot += loss.item()
        if nb % cfg.accum:
            torch.nn.utils.clip_grad_norm_(trainable, cfg.clip)
            opt.step()
            opt.zero_grad(set_to_none=True)

        wer, cer = evaluate(head, bb, ddl, dv, flen, dev, i2c, blank, unk)
        rec = {"epoch": ep, "loss": tot / max(1, nb), "wer": wer, "cer": cer,
               "secs": time.perf_counter() - t0,
               "w": head.weights().detach().cpu().numpy().round(4).tolist()}
        if dev == "cuda":
            rec["vram_gb"] = torch.cuda.max_memory_allocated() / 1e9
        hist.append(rec)
        with hist_path.open("a") as f:
            f.write(json.dumps(rec) + "\n")
        log(f"  e{ep:>3} | loss {rec['loss']:.3f} | {rec['secs']:.0f}s | "
            f"VAL wer {wer * 100:.2f} cer {cer * 100:.2f}")
        sch.step(cer)

        if cer < best * 0.995:
            best, best_ep = cer, ep
            torch.save(head.state_dict(), run_dir / "head.pt")
            torch.save({k: v.detach().cpu().clone() for k, v in bb.state_dict().items()
                        if "lora_" in k}, run_dir / "adapter.pt")
            log(f"     [BEST] {cer * 100:.2f}%")

        # per-epoch IMMUTABLE checkpoint -- an ~8h unattended run must survive
        # a disconnect; a single overwritten last.pt is one bad write from
        # losing everything, so every epoch also gets its own snapshot file.
        ckpt = {"head": head.state_dict(),
                "adapter": {k: v.detach().cpu().clone() for k, v in bb.state_dict().items()
                           if "lora_" in k},
                "opt": opt.state_dict(), "sch": sch.state_dict(),
                "epoch": ep, "best": best, "best_ep": best_ep}
        torch.save(ckpt, run_dir / f"ep{ep:03d}.pt")
        torch.save(ckpt, last_path)  # resumable pointer to "latest"

        if ep - best_ep >= cfg.stop_patience:
            log("[STOP] no improvement, early stopping")
            break

    summary = {"run": cfg.run, "best_cer": best, "best_epoch": best_ep,
               "epochs_done": hist[-1]["epoch"] if hist else 0,
               "vram_peak_gb": torch.cuda.max_memory_allocated() / 1e9 if dev == "cuda" else None,
               "sec_per_epoch": float(np.median([h["secs"] for h in hist])) if hist else None,
               "final_layer_weights": hist[-1]["w"] if hist else None}
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    log(f"[DONE] {cfg.run}: best CER {best * 100:.2f}% @ epoch {best_ep}")
    return summary


def parse_layers(s: str) -> tuple:
    if "-" in s and "," not in s:
        a, b = s.split("-")
        return tuple(range(int(a), int(b) + 1))
    return tuple(int(x) for x in s.split(","))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", required=True)
    ap.add_argument("--cache-dir", required=True)
    ap.add_argument("--out", default="/marimo/runs")
    ap.add_argument("--ws", default="9,10,11,12")
    ap.add_argument("--lora-layers", default="1-12")
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--batch", type=int, default=64)
    ap.add_argument("--accum", type=int, default=4)
    ap.add_argument("--lr-scale", type=float, default=1.0)
    ap.add_argument("--hours-subset", type=float, default=None)
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--no-aug", action="store_true")
    ap.add_argument("--noise-dir", default=None)
    ap.add_argument("--rir-dir", default=None)
    args = ap.parse_args()

    cfg = Cfg(run=args.run, ws=parse_layers(args.ws), lora_layers=parse_layers(args.lora_layers),
              epochs=args.epochs, batch=args.batch, accum=args.accum, lr_scale=args.lr_scale,
              hours_subset=args.hours_subset, workers=args.workers, aug_on=not args.no_aug,
              noise_dir=args.noise_dir, rir_dir=args.rir_dir)
    train_one(cfg, Path(args.out), Path(args.cache_dir))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile eval_asr.py
# /// script
# requires-python = ">=3.11"
# dependencies = [
#     "torch", "torchaudio", "transformers>=4.44", "peft>=0.11", "jiwer",
#     "psutil", "numpy", "datasets==5.0.0", "soundfile==0.14.0",
#     "pyctcdecode", "kenlm",
# ]
# [tool.uv.sources]
# torch = { index = "pytorch-cu128" }
# torchaudio = { index = "pytorch-cu128" }
# [[tool.uv.index]]
# name = "pytorch-cu128"
# url = "https://download.pytorch.org/whl/cu128"
# explicit = true
# ///
"""
ASR -- 300h retrain, headline evaluation.

TWO TEST COLUMNS (not two separate tables) for every system:
    dev-clean    -- comparability anchor; both the 100h baseline and the
                    300h model are scored here so the new number is directly
                    comparable to the published 10.1% / 5.1% WER.
    L2-ARCTIC    -- held-out OOD accent test. NEVER seen in training
                    (enforced upstream by prepare_data.assert_no_l2arctic).

Each column reports greedy AND +KenLM separately -- the KenLM gain is
expected to SHRINK out-of-domain (the LM was built on LibriSpeech text), and
that shrinkage is reported as a real finding, not hidden by only reporting
one decode mode. Whisper (base/small/medium) is re-run on BOTH columns as
the external reference, same protocol as whisper_bench.ipynb.

Efficiency stats follow the existing project's convention:
    CPU RTF   = wall_time_on_cpu / audio_duration_s     (psutil process)
    GPU RTF   = wall_time_on_gpu / audio_duration_s
    peak RAM  = psutil RSS delta (CPU) / torch.cuda.max_memory_allocated (GPU)
"""

from __future__ import annotations

import argparse
import json
import re
import sys
import time
from itertools import groupby
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
from prepare_data import build_vocab, normalize_text

_NORM_RE = re.compile(r"[^A-Z' ]+")


def wer_normalize(s: str) -> str:
    """Common normalizer applied to BOTH ref and hyp before jiwer -- same
    convention as kenlm_grid.py's normalize(), so WER isn't inflated by a
    spurious ref/hyp mismatch in punctuation handling."""
    s = s.upper().replace("|", " ")
    s = _NORM_RE.sub(" ", s)
    return " ".join(s.split())


def log(*a):
    print(*a, flush=True)


# ============================================================================
# 1 . Test set loaders
# ============================================================================


def load_devclean(limit=None):
    from datasets import load_dataset

    ds = load_dataset("openslr/librispeech_asr", "clean", split="validation")
    rows = []
    for i in range(len(ds) if limit is None else min(limit, len(ds))):
        r = ds[i]
        rows.append({"audio": r["audio"], "text": r["text"]})
    log(f"[devclean] {len(rows)} utterances")
    return rows


def load_l2arctic(limit=None):
    """L2-ARCTIC -- OOD accent test set. Loaded HERE ONLY, at eval time,
    never in prepare_data.py / train_asr.py. If this repo id is wrong for
    the actual HF mirror in use, that is the one thing to fix before running
    eval -- everything downstream (normalisation, WER) is repo-id agnostic."""
    from datasets import load_dataset

    ds = None
    for repo in ("babels/l2-arctic", "keithito/l2-arctic", "l2-arctic"):
        try:
            ds = load_dataset(repo, split="train")
            log(f"[l2arctic] loaded from {repo}")
            break
        except Exception as e:
            log(f"[l2arctic] {repo} failed ({type(e).__name__})")
    if ds is None:
        raise RuntimeError("Could not load L2-ARCTIC from any known repo id -- "
                            "verify the correct HF repo id before running eval.")
    rows = []
    for i in range(len(ds) if limit is None else min(limit, len(ds))):
        r = ds[i]
        text = r.get("text") or r.get("transcript") or r.get("sentence")
        rows.append({"audio": r["audio"], "text": text})
    log(f"[l2arctic] {len(rows)} utterances")
    return rows


# ============================================================================
# 2 . Our model: greedy + KenLM decode
# ============================================================================


def _decode_audio_array(cell, sr_target=16000):
    import numpy as np

    w = np.asarray(cell["array"], dtype=np.float32)
    sr = cell["sampling_rate"]
    if int(sr) != sr_target:
        w = np.interp(np.linspace(0, len(w) - 1, int(len(w) * sr_target / sr)),
                      np.arange(len(w)), w).astype(np.float32)
    return w


def greedy_decode(logits, vocab):
    blank, unk = vocab["[PAD]"], vocab["[UNK]"]
    i2c = {i: c for c, i in vocab.items()}
    ids = logits.argmax(-1)
    out = [i2c[k] for k, _ in groupby(ids.tolist()) if k not in (blank, unk)]
    return "".join(out).replace("|", " ").strip()


def load_our_model(run_dir: Path, device: str):
    import torch
    import torch.nn as nn
    from transformers import HubertModel
    from peft import LoraConfig, inject_adapter_in_model

    cfg = json.loads((run_dir / "config.json").read_text())
    ws, lora_layers = cfg["ws"], cfg["lora_layers"]
    hid = cfg.get("hid", 768)

    bb = HubertModel.from_pretrained("utter-project/mHuBERT-147")
    lora_cfg = LoraConfig(r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"], lora_dropout=0.0,
                          target_modules=["q_proj", "v_proj"], bias="none",
                          layers_to_transform=[i - 1 for i in lora_layers])
    bb = inject_adapter_in_model(lora_cfg, bb)
    bb.load_state_dict(torch.load(run_dir / "adapter.pt", map_location=device), strict=False)
    bb = bb.to(device).eval()

    vocab = build_vocab()

    class Head(nn.Module):
        def __init__(self, n, dim, V):
            super().__init__()
            self.layer_w = nn.Parameter(torch.zeros(n))
            self.net = nn.Sequential(nn.Linear(dim, dim), nn.ELU(), nn.Dropout(0.0),
                                     nn.Linear(dim, V))

        def forward(self, x):
            w = self.layer_w.softmax(0)
            return self.net((x * w[None, None, :, None]).sum(2))

    head = Head(len(ws), hid, len(vocab)).to(device)
    head.load_state_dict(torch.load(run_dir / "head.pt", map_location=device))
    head.eval()
    return bb, head, ws, vocab


def run_our_model(rows, run_dir: Path, lm_path: str | None, device: str) -> dict:
    import torch
    import psutil

    bb, head, ws, vocab = load_our_model(run_dir, device)
    blank, unk = vocab["[PAD]"], vocab["[UNK]"]
    flen = bb._get_feat_extract_output_lengths

    decoder = None
    if lm_path:
        from pyctcdecode import build_ctcdecoder

        # inlined from kenlm_grid.py's vocab_to_labels -- kept local instead
        # of importing across the _Staj/ vs _Staj/asr/ directory boundary,
        # since these notebooks are meant to be standalone (kenlm_grid.py's
        # own header rule: no imports across scripts)
        def _vocab_to_labels(v):
            labels = [""] * len(v)
            for tok, i in v.items():
                if tok == "|":
                    labels[i] = " "
                elif tok == "[PAD]":
                    labels[i] = ""
                elif tok == "[UNK]":
                    labels[i] = "?"
                else:
                    labels[i] = tok
            return labels

        decoder = build_ctcdecoder(_vocab_to_labels(vocab), kenlm_model_path=lm_path,
                                   alpha=0.5, beta=1.0)

    proc = psutil.Process()
    refs, hyps_greedy, hyps_lm = [], [], []
    total_audio_s, t_greedy, t_lm = 0.0, 0.0, 0.0
    peak_rss = proc.memory_info().rss

    with torch.no_grad():
        for r in rows:
            w = _decode_audio_array(r["audio"])
            total_audio_s += len(w) / 16000.0
            X = torch.from_numpy(w).unsqueeze(0).to(device)
            am = torch.ones_like(X, dtype=torch.long)

            t0 = time.perf_counter()
            with torch.autocast(device_type=device, dtype=torch.bfloat16):
                o = bb(X, attention_mask=am, output_hidden_states=True)
                hs = torch.stack([o.hidden_states[L] for L in ws], 2)
                logits = head(hs.float())[0]
            if device == "cuda":
                torch.cuda.synchronize()
            t_greedy += time.perf_counter() - t0

            probs = logits.log_softmax(-1)
            hyps_greedy.append(wer_normalize(greedy_decode(probs.cpu(), vocab)))
            refs.append(wer_normalize(r["text"]))

            if decoder is not None:
                t1 = time.perf_counter()
                hyp = decoder.decode(probs.cpu().numpy())
                t_lm += (time.perf_counter() - t1) + (t_greedy)  # includes forward pass
                hyps_lm.append(wer_normalize(hyp))

            peak_rss = max(peak_rss, proc.memory_info().rss)

    import jiwer

    out = {
        "greedy": {"wer": jiwer.wer(refs, hyps_greedy), "cer": jiwer.cer(refs, hyps_greedy),
                   "rtf": t_greedy / max(total_audio_s, 1e-6)},
    }
    if decoder is not None:
        out["kenlm"] = {"wer": jiwer.wer(refs, hyps_lm), "cer": jiwer.cer(refs, hyps_lm),
                        "rtf": t_lm / max(total_audio_s, 1e-6)}
    out["peak_ram_mb"] = (peak_rss - psutil.Process().memory_info().rss + peak_rss) / 1e6
    if device == "cuda":
        out["peak_gpu_gb"] = torch.cuda.max_memory_allocated() / 1e9
    return out


# ============================================================================
# 3 . Whisper baseline -- same protocol, both columns
# ============================================================================


def run_whisper(rows, model_name: str, device: str) -> dict:
    import torch
    import psutil
    import jiwer
    from transformers import WhisperForConditionalGeneration, WhisperProcessor

    proc_model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device).eval()
    processor = WhisperProcessor.from_pretrained(model_name)
    ps = psutil.Process()

    refs, hyps = [], []
    total_audio_s, t_total = 0.0, 0.0
    with torch.no_grad():
        for r in rows:
            w = _decode_audio_array(r["audio"])
            total_audio_s += len(w) / 16000.0
            inputs = processor(w, sampling_rate=16000, return_tensors="pt").to(device)
            t0 = time.perf_counter()
            ids = proc_model.generate(inputs["input_features"], language="en", task="transcribe")
            if device == "cuda":
                torch.cuda.synchronize()
            t_total += time.perf_counter() - t0
            hyp = processor.batch_decode(ids, skip_special_tokens=True)[0]
            refs.append(wer_normalize(r["text"]))
            hyps.append(wer_normalize(hyp))

    out = {"wer": jiwer.wer(refs, hyps), "cer": jiwer.cer(refs, hyps),
           "rtf": t_total / max(total_audio_s, 1e-6),
           "peak_ram_mb": ps.memory_info().rss / 1e6}
    if device == "cuda":
        out["peak_gpu_gb"] = torch.cuda.max_memory_allocated() / 1e9
    return out


# ============================================================================
# 4 . Orchestration
# ============================================================================


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run-dir", required=True, help="trained run dir (has config.json/head.pt/adapter.pt)")
    ap.add_argument("--lm", default=None, help="path to KenLM .arpa (omit for greedy-only)")
    ap.add_argument("--limit", type=int, default=None, help="cap rows per test set (debug)")
    ap.add_argument("--whisper", default="openai/whisper-base,openai/whisper-small,openai/whisper-medium")
    ap.add_argument("--out", default=None)
    args = ap.parse_args()

    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dev_rows = load_devclean(args.limit)
    l2_rows = load_l2arctic(args.limit)

    results = {"dev-clean": {}, "l2-arctic": {}}

    log("[eval] scoring FINAL model on dev-clean...")
    results["dev-clean"]["FINAL_300h"] = run_our_model(dev_rows, Path(args.run_dir), args.lm, device)
    log("[eval] scoring FINAL model on l2-arctic...")
    results["l2-arctic"]["FINAL_300h"] = run_our_model(l2_rows, Path(args.run_dir), args.lm, device)

    for wm in args.whisper.split(","):
        tag = wm.split("/")[-1]
        log(f"[eval] scoring {tag} on dev-clean...")
        results["dev-clean"][tag] = run_whisper(dev_rows, wm, device)
        log(f"[eval] scoring {tag} on l2-arctic...")
        results["l2-arctic"][tag] = run_whisper(l2_rows, wm, device)

    log(json.dumps(results, indent=2))
    if args.out:
        Path(args.out).write_text(json.dumps(results, indent=2))
        log(f"[eval] written to {args.out}")


if __name__ == "__main__":
    main()


## 3 · MUSAN / DEMAND / OpenSLR-28 RIR download

Fetched once into `NOISE_DIR` (MUSAN noise subset + DEMAND, ~6h each) and
`RIR_DIR` (OpenSLR-28). These feed `augment.py`'s `AudioBank`, which loads
them into RAM once (~1.4 GB as fp32) and samples batches on GPU.

In [ ]:
# Best-effort direct downloads (openslr.org mirrors, same convention
# kenlm_grid.py already uses for the LM archive). Verify URLs still
# resolve before a real run — no network access was available while writing
# this notebook to confirm them.
import urllib.request, tarfile, zipfile, shutil

def fetch_and_extract(url, dest_dir, kind="tar"):
    dest_dir.mkdir(parents=True, exist_ok=True)
    local = dest_dir / Path(url).name
    if not local.exists():
        print(f"downloading {url} -> {local}")
        urllib.request.urlretrieve(url, local)
    if kind == "tar" and local.suffixes[-2:] in ([".tar", ".gz"], [".tgz"]):
        with tarfile.open(local) as t:
            t.extractall(dest_dir)
    elif kind == "zip":
        with zipfile.ZipFile(local) as z:
            z.extractall(dest_dir)

# MUSAN (noise subset only — speech/music not needed for this augmentation)
fetch_and_extract("https://www.openslr.org/resources/17/musan.tar.gz", NOISE_DIR / "musan", "tar")

# OpenSLR-28 simulated RIRs
fetch_and_extract("https://www.openslr.org/resources/28/rirs_noises.zip", RIR_DIR, "zip")

# DEMAND — distributed as one zip per noise environment on Zenodo (record
# 1227121); pull a handful of 16kHz channel-1 files, enough for ~6h total.
DEMAND_ENVS = ["DKITCHEN", "DWASHING", "NPARK", "OMEETING", "PSTATION", "TBUS"]
for env in DEMAND_ENVS:
    url = f"https://zenodo.org/record/1227121/files/{env}_16k.zip"
    try:
        fetch_and_extract(url, NOISE_DIR / "demand" / env, "zip")
    except Exception as e:
        print(f"[demand] {env} failed ({type(e).__name__}) — verify the Zenodo record id/layout")

print("noise/RIR banks ready (inspect NOISE_DIR / RIR_DIR before training if any step above failed)")


## 4 · Build the 300h manifest

Per-corpus stats (hours kept, filter drop counts, accent histogram, OOV
drop rate) are logged by each `prepare_data.py` step and written alongside
the manifests in `DATA_DIR`.

In [ ]:
run("prepare_data.py", "--corpus", "librispeech", "--out", DATA_DIR, "--cache", CACHE_DIR)
run("prepare_data.py", "--corpus", "common_voice", "--out", DATA_DIR, "--cache", CACHE_DIR)
run("prepare_data.py", "--corpus", "ami", "--out", DATA_DIR, "--cache", CACHE_DIR)
run("prepare_data.py", "--corpus", "vctk", "--out", DATA_DIR, "--cache", CACHE_DIR)
run("prepare_data.py", "--combine", "--out", DATA_DIR, "--cache", CACHE_DIR)


In [ ]:
import json
stats = json.loads((DATA_DIR / "manifest_combined.stats.json").read_text())
print(json.dumps(stats, indent=2))
assert 280 <= stats["total_hours"] <= 320, "combined manifest is far from the 300h target — check per-corpus logs above"


In [ ]:
# L2-ARCTIC leakage check, one more time, at the notebook level (belt and
# suspenders on top of prepare_data.assert_no_l2arctic, which already ran
# on every write above): scan the actual manifest file bytes on disk.
combined_text = (DATA_DIR / "manifest_combined.jsonl").read_text().lower()
for marker in ("l2-arctic", "l2_arctic", "l2arctic"):
    assert marker not in combined_text, f"L2-ARCTIC marker '{marker}' found in combined manifest!"
print("confirmed: no L2-ARCTIC marker anywhere in manifest_combined.jsonl")


In [ ]:
run("build_cache.py", "--manifest", DATA_DIR / "manifest_combined.jsonl", "--cache", CACHE_DIR)
import glob
cache_dirs = sorted(glob.glob(str(CACHE_DIR / "combined_*")))
COMBINED_CACHE = cache_dirs[-1]
print("using cache:", COMBINED_CACHE)


## 5 · 50h probe — does more diverse data shift the optimal weighted-sum layer?

Run **with augmentation enabled** (tuning it off would select a config for
a training condition never actually used). Three arms, same LoRA scope
(layers 1–12), different weighted-sum reads:

| arm | WS layers | hypothesis |
|---|---|---|
| `probe_control` | [9,10,11,12] | current FINAL config |
| `probe_lowerA` | [5,6,7,8] | lower layers = more phonetic; accent/noise robustness is a phonetic problem |
| `probe_lowerB` | [7,8,9,10] | middle ground |

High LR (`--lr-scale 3.0`), short budget (6 epochs) — this is a fast probe,
not a final model.

In [ ]:
PROBE_ARMS = {
    "probe_control": "9,10,11,12",
    "probe_lowerA": "5,6,7,8",
    "probe_lowerB": "7,8,9,10",
}
for run_name, ws in PROBE_ARMS.items():
    run("train_asr.py",
        "--run", run_name, "--cache-dir", COMBINED_CACHE, "--out", RUNS_DIR,
        "--ws", ws, "--lora-layers", "1-12",
        "--hours-subset", "50", "--epochs", "6", "--lr-scale", "3.0",
        "--noise-dir", NOISE_DIR, "--rir-dir", RIR_DIR)


In [ ]:
import json
probe_results = {}
for run_name in PROBE_ARMS:
    s = json.loads((RUNS_DIR / run_name / "summary.json").read_text())
    probe_results[run_name] = s
    print(f"{run_name:16s} WS={PROBE_ARMS[run_name]:12s} best CER {s['best_cer']*100:5.2f}% "
          f"@ep{s['best_epoch']} | layer weights {s['final_layer_weights']}")

winner = min(probe_results, key=lambda k: probe_results[k]["best_cer"])
print(f"\nprobe winner: {winner} (WS={PROBE_ARMS[winner]})")


## 6 · Full 300h run

Uses the winning weighted-sum configuration from the probe above (falls
back to the control/FINAL config if the probe result is inconclusive — edit
`FINAL_WS` manually if you want to override the automatic pick).
Per-epoch checkpoints (`ep{N:03d}.pt`) plus a resumable `last.pt` are
written every epoch — required for an unattended ~8h cloud run to survive
a disconnect.

In [ ]:
FINAL_WS = PROBE_ARMS[winner]   # override manually here if desired, e.g. FINAL_WS = "9,10,11,12"
print("training FULL 300h run with WS =", FINAL_WS)

run("train_asr.py",
    "--run", "FINAL_300h", "--cache-dir", COMBINED_CACHE, "--out", RUNS_DIR,
    "--ws", FINAL_WS, "--lora-layers", "1-12",
    "--epochs", "30", "--batch", "64", "--accum", "4",
    "--noise-dir", NOISE_DIR, "--rir-dir", RIR_DIR)


In [ ]:
# Re-run this cell alone to resume after a disconnect — train_asr.py picks
# up from last.pt automatically if RUNS_DIR/FINAL_300h/last.pt exists.
run("train_asr.py",
    "--run", "FINAL_300h", "--cache-dir", COMBINED_CACHE, "--out", RUNS_DIR,
    "--ws", FINAL_WS, "--lora-layers", "1-12",
    "--epochs", "30", "--batch", "64", "--accum", "4",
    "--noise-dir", NOISE_DIR, "--rir-dir", RIR_DIR)


## 7 · Headline evaluation — dev-clean × L2-ARCTIC, greedy + KenLM

Two test columns for every system (our FINAL_300h model AND Whisper
base/small/medium), greedy and +KenLM reported separately in both. The
KenLM gain is expected to shrink on L2-ARCTIC (the LM was built on
LibriSpeech text) — that shrinkage is a real finding, reported here rather
than hidden by only showing one decode mode.

In [ ]:
# Reuse the existing OpenSLR 3-gram pruned LM (same one kenlm_grid.py uses)
# so the KenLM column is directly comparable to the published 5.1% number.
LM_PATH = "/marimo/lm_work/3-gram.pruned.1e-7.arpa"
import os
if not os.path.exists(LM_PATH):
    os.makedirs(os.path.dirname(LM_PATH), exist_ok=True)
    run("-c", f'''
import urllib.request, gzip, shutil
urllib.request.urlretrieve("https://www.openslr.org/resources/11/3-gram.pruned.1e-7.arpa.gz", "{LM_PATH}.gz")
with gzip.open("{LM_PATH}.gz") as fi, open("{LM_PATH}", "wb") as fo:
    shutil.copyfileobj(fi, fo)
''')

run("eval_asr.py",
    "--run-dir", RUNS_DIR / "FINAL_300h",
    "--lm", LM_PATH,
    "--out", RUNS_DIR / "FINAL_300h" / "eval_results.json")


In [ ]:
import json
results = json.loads((RUNS_DIR / "FINAL_300h" / "eval_results.json").read_text())

print(f"{'system':16s} {'devclean greedy':>16s} {'devclean +LM':>14s} "
      f"{'l2arctic greedy':>16s} {'l2arctic +LM':>14s}")
for sysname in results["dev-clean"]:
    dc = results["dev-clean"][sysname]
    l2 = results["l2-arctic"][sysname]
    dc_g = f"{dc.get('wer', dc.get('greedy', {}).get('wer', float('nan')))*100:.1f}"
    dc_lm = f"""{dc.get('kenlm', {}).get('wer', float('nan'))*100:.1f}""" if 'kenlm' in dc else "-"
    l2_g = f"{l2.get('wer', l2.get('greedy', {}).get('wer', float('nan')))*100:.1f}"
    l2_lm = f"""{l2.get('kenlm', {}).get('wer', float('nan'))*100:.1f}""" if 'kenlm' in l2 else "-"
    print(f"{sysname:16s} {dc_g:>16s} {dc_lm:>14s} {l2_g:>16s} {l2_lm:>14s}")


## 8 · Efficiency — CPU/GPU RTF, peak RAM

Same convention as `cpu_bench.ipynb` / `whisper_bench.ipynb`:
RTF = wall time / audio duration; peak RAM via `psutil` (CPU) and
`torch.cuda.max_memory_allocated()` (GPU). These are already captured per
system inside `eval_results.json` (`rtf`, `peak_ram_mb`, `peak_gpu_gb`
fields) from the GPU eval run above; run the same `eval_asr.py` step with
`CUDA_VISIBLE_DEVICES=""` for the CPU-side numbers.

In [ ]:
import os
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = ""
import subprocess
subprocess.run([PY311, "eval_asr.py",
                "--run-dir", str(RUNS_DIR / "FINAL_300h"),
                "--lm", LM_PATH, "--limit", "200",   # CPU is slow -- cap rows for the RTF measurement
                "--out", str(RUNS_DIR / "FINAL_300h" / "eval_results_cpu.json")],
               cwd=str(ASR_DIR), env=env, check=True)

cpu_results = json.loads((RUNS_DIR / "FINAL_300h" / "eval_results_cpu.json").read_text())
print(json.dumps({k: v.get("FINAL_300h", {}) for k, v in cpu_results.items()}, indent=2))


## 9 · Notes / caveats

- The probe's winning arm is picked automatically by lowest dev CER;
  inspect `probe_results` before trusting it blindly on a noisy 6-epoch run.
- `eval_asr.py`'s L2-ARCTIC loader tries a short list of HF repo ids — fix
  the correct one first if all of them fail (no network access was
  available while writing this notebook to confirm the right id).
- Common Voice's exact repo layout (`transcript/en/*` vs `audio/en/**` vs
  something else) was not verified against a live snapshot; if
  `download_common_voice_en` pulls unexpected extra files, tighten
  `allow_patterns` in `prepare_data.py` before re-running.
- See `asr/README.md` for the full list of design decisions and what could
  not be verified without a GPU/network in the environment this was written in.
